# SensorFM — throughput ladder + scaling study (Tasks 3.1–3.3)

**Accelerator: GPU T4 x2**. Settings: **Internet: On**. Secrets (Add-ons → Secrets): `GH_TOKEN`. Input: **Add Input → Your Datasets → `sensorfm-data`**.

Budget ≈ 1.1 session-hours. Results land in `/kaggle/working/results`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['GH_TOKEN'] = s.get_secret('GH_TOKEN')
try:
    os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret (optional)')

In [ ]:
%%bash
set -e
cd /kaggle/working
rm -rf sensorfm-src && git clone -q https://$GH_TOKEN@github.com/DhrumilPanchal10/sensorfm.git sensorfm-src
cd sensorfm-src && git log --oneline -1

In [ ]:
%%bash
set -e
cd /kaggle/working/sensorfm-src
CUDA_VISIBLE_DEVICES=0 MODEL=s20m bash kaggle/run_ladder.sh 2>&1 | grep -v Warning

In [ ]:
%%bash
set -e
cd /kaggle/working/sensorfm-src
bash kaggle/run_scaling.sh 2>&1 | grep -v Warning
source kaggle/gpu_setup.sh >/dev/null && python scripts/build_report.py && cat docs/reports/status.md
cp -r docs/reports /kaggle/working/reports

In [ ]:
# Keep the saved output small: results/ and runs/ were copied to /kaggle/working already.
import shutil; shutil.rmtree('/kaggle/working/sensorfm-src', ignore_errors=True)
!du -sh /kaggle/working/* 2>/dev/null